# 04 — Evaluation, Error Analysis & Discussion
### NutriFit-AI

Produces every figure and table the **Analysis and Discussion** chapter needs.

**The headline result is not the one the proposal predicted**, and this notebook
gathers the evidence that explains why. That honesty — with data to back it —
is worth more marks than an unexamined "Random Forest won".

**Runtime:** CPU (except the optional §6 GPU benchmark).

In [ ]:
# ============================================================
# SETUP - run this first in every notebook
# ============================================================
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/NutriFit-AI")
    if not PROJECT.exists():
        raise FileNotFoundError(
            f"{PROJECT} not found.\n"
            "Upload the whole NutriFit-AI folder to the ROOT of your Google Drive "
            "(My Drive/NutriFit-AI), then re-run this cell."
        )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "scikit-learn>=1.4", "pandas>=2.1", "joblib>=1.3", "seaborn>=0.13"],
        check=False,
    )
else:
    PROJECT = Path.cwd()
    while not (PROJECT / "ml" / "nutrifit").exists() and PROJECT != PROJECT.parent:
        PROJECT = PROJECT.parent

sys.path.insert(0, str(PROJECT / "ml"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nutrifit
from nutrifit import config, data, foods, labels, nutrition, planner, preprocessing, recommender, training

for directory in (config.PROCESSED_DIR, config.ARTIFACTS_DIR, config.FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["savefig.bbox"] = "tight"
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

print(f"nutrifit  {nutrifit.__version__}")
print(f"project   {PROJECT}")
print(f"data/raw  {config.RAW_DIR}")
print(f"figures   {config.FIGURES_DIR}")
print(f"in colab  {IN_COLAB}")

In [ ]:
def savefig(name):
    """Save the current figure into reports/figures/ for the dissertation."""
    path = config.FIGURES_DIR / f"{name}.png"
    plt.savefig(path)
    print(f"saved {path}")

In [ ]:
import joblib
labelled = pd.read_csv(config.USERS_PROCESSED)
calorie_bundle = joblib.load(config.ARTIFACTS_DIR / "_calorie_bundle.pkl")
protein_bundle = joblib.load(config.ARTIFACTS_DIR / "_protein_bundle.pkl")
bundles = {"calorie_target": calorie_bundle, "protein_target": protein_bundle}
print("Loaded training bundles.")

## 1. Headline metrics — all four required measures

In [ ]:
comparison = training.comparison_table([calorie_bundle, protein_bundle])
display(comparison)

### 1.1 Against the non-ML formula baseline and the noise ceiling

Three reference points make the numbers interpretable:

* **Formula baseline** — what a plain calculator achieves (it knows the activity
  multiplier; the models do not).
* **Theoretical R² ceiling** — the maximum achievable given injected noise.
* **Model R²** — where each model actually lands.

In [ ]:
rows = []
for target, bundle in bundles.items():
    baseline = training.formula_baseline_metrics(labelled, target)
    ceiling = labels.theoretical_r2_ceiling(labelled, target)
    rows.append({"target": target, "reference": "Formula baseline (knows PAL)",
                 "MAE": round(baseline["mae"], 2), "R2": round(baseline["r2"], 4)})
    rows.append({"target": target, "reference": "Theoretical R2 ceiling",
                 "MAE": np.nan, "R2": round(ceiling, 4)})
    for name, result in bundle["results"].items():
        rows.append({"target": target, "reference": result.model_name,
                     "MAE": round(result.test_metrics["mae"], 2),
                     "R2": round(result.test_metrics["r2"], 4)})
display(pd.DataFrame(rows))

## 2. Predicted vs actual, and residual structure

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for col, (target, bundle) in enumerate(bundles.items()):
    y_test = bundle["split"]["y_test"].to_numpy(float)
    for row, model in enumerate(("LinearRegression", "RandomForest")):
        ax = axes[row, col]
        pred = bundle["predictions"][model]
        ax.scatter(y_test, pred, s=14, alpha=0.5)
        lo, hi = float(min(y_test.min(), pred.min())), float(max(y_test.max(), pred.max()))
        ax.plot([lo, hi], [lo, hi], "r--", lw=1.5)
        r2 = bundle["results"][model].test_metrics["r2"]
        mae = bundle["results"][model].test_metrics["mae"]
        ax.set_title(f"{target} — {model}\nR2={r2:.4f}  MAE={mae:.2f}")
        ax.set_xlabel("actual"); ax.set_ylabel("predicted")
plt.tight_layout()
savefig("eval_predicted_vs_actual")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, (target, bundle) in zip(axes, bundles.items()):
    for model, colour in (("LinearRegression", "steelblue"), ("RandomForest", "darkorange")):
        residual = bundle["split"]["y_test"].to_numpy(float) - bundle["predictions"][model]
        sns.kdeplot(residual, ax=ax, label=model, fill=True, alpha=0.3, color=colour)
    ax.axvline(0, color="black", lw=1, ls="--")
    ax.set_title(f"Residual distribution — {target}"); ax.set_xlabel("actual - predicted"); ax.legend()
plt.tight_layout()
savefig("eval_residual_distributions")
plt.show()

### 2.1 Is error concentrated in any subgroup?

A fairness / robustness check: if one goal or sex carries most of the error,
that is a real limitation and belongs in the report.

In [ ]:
for target, bundle in bundles.items():
    frame = training.residual_frame(bundle, "RandomForest")
    print(f"\n=== {target} — mean absolute error by subgroup ===")
    display(pd.concat([
        frame.groupby("fitness_goal")["abs_error"].agg(["mean","std","count"]).round(2),
        frame.groupby("gender")["abs_error"].agg(["mean","std","count"]).round(2),
    ], keys=["fitness_goal","gender"]))

## 3. Why does Linear Regression beat Random Forest here?

The proposal expected Random Forest to win. It does not. Before accepting that,
two competing explanations must be ruled out:

1. **Under-tuning** — the search space starved the forest.
2. **Data scarcity** — a forest approximating a smooth, largely additive
   function with axis-aligned splits needs far more than ~1000 samples.

The learning curve below distinguishes them. If the gap **narrows as n grows**,
the cause is data scarcity, not tuning.

In [ ]:
curves = {}
for target in bundles:
    print(f"computing learning curve for {target} ...")
    curves[target] = training.learning_curve_comparison(labelled, target, seed=config.RANDOM_SEED)
    curves[target].to_csv(config.REPORTS_DIR / f"learning_curve_{target}.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, (target, curve) in zip(axes, curves.items()):
    for model, marker in (("Linear Regression", "o"), ("Random Forest", "s")):
        subset = curve[curve["model"] == model]
        ax.errorbar(subset["n_samples"], subset["cv_mae_mean"], yerr=subset["cv_mae_std"],
                    marker=marker, capsize=3, label=model, lw=2)
    ax.set_xlabel("training samples"); ax.set_ylabel("cross-validated MAE")
    ax.set_title(f"Learning curve — {target}"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
savefig("eval_learning_curves")
plt.show()

for target, curve in curves.items():
    pivot = curve.pivot(index="n_samples", columns="model", values="cv_mae_mean")
    pivot["gap (RF - LR)"] = pivot["Random Forest"] - pivot["Linear Regression"]
    print(f"\n=== {target} ===")
    display(pivot.round(3))

### 3.1 Reading the learning curve

Look at the **gap column** in the tables above:

* Linear Regression's error is **flat** from roughly n≈200 onward — it has
  already extracted everything its functional form can represent.
* Random Forest's error **falls monotonically** and the gap **shrinks
  steadily** with more data.

That is the signature of a **high-bias, data-starved** estimator, not a
mis-tuned one. Trees approximate a smooth function with piecewise-constant
steps; closing that approximation error requires many more samples than 973.

**What to write in the report:**

> *Contrary to the expectation stated in the proposal, Linear Regression
> outperformed Random Forest on both targets. The learning-curve analysis
> identifies the cause as dataset size rather than hyper-parameter selection:
> Random Forest's cross-validated MAE decreases monotonically with training-set
> size while Linear Regression's plateaus early, and the gap narrows
> consistently. This is consistent with the bias–variance characteristics of the
> two model families — the underlying physiological relationship (Mifflin–St
> Jeor / Katch–McArdle scaled by an activity multiplier) is smooth and close to
> linear in the model's raw inputs, which is exactly the regime in which a
> correctly-specified linear model is efficient and an axis-aligned ensemble is
> not. Random Forest would be expected to overtake given a substantially larger
> dataset; with 973 records it does not, and the deployed model is selected on
> cross-validated MAE accordingly.*

This directly satisfies the marking criteria for *consideration of alternative
approaches* and *understanding of application and potential limitations*.

## 4. Feature importance and coefficients

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, (target, bundle) in zip(axes, bundles.items()):
    top = bundle["importances"].head(12)
    sns.barplot(data=top, y="feature", x="importance", ax=ax)
    ax.set_title(f"Random Forest feature importance — {target}")
plt.tight_layout()
savefig("eval_feature_importance")
plt.show()

for target, bundle in bundles.items():
    print(f"\n=== {target} — Linear Regression coefficients (standardised inputs) ===")
    display(bundle["coefficients"].head(12).round(3))

### 4.1 Do the drivers match nutrition science?

Check these against domain expectation and discuss any mismatch:

* **Calories** — `weight_kg` / `body_fat_pct` should dominate (they set BMR via
  lean mass), followed by `workout_frequency` and `session_duration_h` (they set
  the activity multiplier), then `fitness_goal`.
* **Protein** — `weight_kg` should overwhelmingly dominate, because the target
  is literally `bodyweight × goal coefficient`. `fitness_goal` should be the
  clear second.

Because inputs are standardised, the linear coefficients are directly
comparable in magnitude — state that when you quote them.

## 5. Recommendation engine and meal planner evaluation

The recommender has no ground-truth labels, so it is evaluated on **macro
accuracy** and **variety** instead — the equivalent of MAE/R² for this component.

In [ ]:
catalogue = foods.add_derived_features(pd.read_csv(config.FOODS_PROCESSED))
engine = recommender.MealRecommender(catalogue, random_state=config.RANDOM_SEED)
print(f"Catalogue: {len(catalogue)} items across {engine.available_slots()}")

scenarios = [("fat_loss", 1900, 145), ("maintenance", 2400, 130), ("muscle_gain", 3000, 175)]
rows = []
for goal, kcal, protein in scenarios:
    sim = recommender.evaluate_recommender(engine, kcal, protein, goal, n_days=30,
                                           seed=config.RANDOM_SEED)
    rows.append({
        "goal": goal, "calorie_target": kcal, "protein_target": protein,
        "mean_abs_calorie_error_%": round(sim["calorie_error_pct"].abs().mean(), 2),
        "mean_abs_protein_error_%": round(sim["protein_error_pct"].abs().mean(), 2),
        "unique_foods_used": sim.attrs["unique_meals"],
        "variety_ratio": round(sim.attrs["variety_ratio"], 3),
    })
display(pd.DataFrame(rows))

In [ ]:
plan = planner.generate_meal_plan(engine, calorie_target=2400, protein_target=150,
                                  goal="muscle_gain", weeks=8, seed=config.RANDOM_SEED)
weekly = plan.weekly_summary()
display(weekly)
print(f"\nVariety report: {plan.variety_report()}")
print(f"Repairs applied: {plan.repairs_applied}")
print(f"Within +/-5% tolerance: {plan.within_tolerance()}")

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].bar(weekly["week"], weekly["calorie_error_pct"], color="steelblue")
axes[0].axhline(5, color="crimson", ls="--"); axes[0].axhline(-5, color="crimson", ls="--")
axes[0].set_title("Weekly calorie error vs +/-5% tolerance"); axes[0].set_xlabel("week"); axes[0].set_ylabel("%")
axes[1].bar(weekly["week"], weekly["protein_error_pct"], color="darkorange")
axes[1].axhline(5, color="crimson", ls="--"); axes[1].axhline(-5, color="crimson", ls="--")
axes[1].set_title("Weekly protein error vs +/-5% tolerance"); axes[1].set_xlabel("week"); axes[1].set_ylabel("%")
plt.tight_layout()
savefig("eval_meal_plan_accuracy")
plt.show()

weekly.to_csv(config.REPORTS_DIR / "meal_plan_weekly_summary.csv", index=False)
plan.to_dataframe().to_csv(config.REPORTS_DIR / "meal_plan_sample.csv", index=False)

### 5.1 A sample week, as the user would see it

In [ ]:
sample = plan.to_dataframe()
week1 = sample[sample["week"] == 1]
display(week1.pivot_table(index="day_of_week", columns="meal_slot",
                          values="name", aggfunc=lambda values: " + ".join(values))
        .reindex(list(planner.DAYS_OF_WEEK)))
display(week1.groupby("day_of_week")[["calories","protein_g","carbs_g","fat_g"]]
        .sum().reindex(list(planner.DAYS_OF_WEEK)).round(0))

### 5.2 Distance-metric comparison — Euclidean vs cosine

Justifies the design decision in `recommender.py`.

In [ ]:
rows = []
for metric in ("euclidean", "cosine"):
    engine_m = recommender.MealRecommender(catalogue, metric=metric, random_state=config.RANDOM_SEED)
    sim = recommender.evaluate_recommender(engine_m, 2400, 150, "maintenance",
                                           n_days=30, seed=config.RANDOM_SEED)
    rows.append({"metric": metric,
                 "mean_abs_calorie_error_%": round(sim["calorie_error_pct"].abs().mean(), 2),
                 "mean_abs_protein_error_%": round(sim["protein_error_pct"].abs().mean(), 2),
                 "variety_ratio": round(sim.attrs["variety_ratio"], 3)})
display(pd.DataFrame(rows))
print("\nCosine is scale-invariant: it matches macro *ratios* and ignores portion size,")
print("which is the wrong objective when the goal is to hit an absolute calorie budget.")

## 6. OPTIONAL — GPU benchmark with RAPIDS cuML

**Only run this section if you want a genuine use for the L4 GPU in your report.**
It is a bonus, not a requirement, and it does **not** change the deployed model.

### Careful instructions

1. **Save everything first.** `File → Save a copy in Drive` (or just Ctrl-S).
2. `Runtime → Change runtime type → L4 GPU → Save`.
   **This restarts the runtime and clears all variables.**
3. Re-run the SETUP cell and the load cell at the top of this notebook.
4. Run the two cells below.
5. **When finished:** `Runtime → Disconnect and delete runtime`.
   Do this immediately — an idle GPU session keeps consuming compute units.

Expected cost: roughly **1–2 compute units** for a ~15-minute session. Verify the
current rate in Colab's resources panel before starting.

In [ ]:
# Cell 1 of 2 - install RAPIDS cuML (takes ~5 minutes; GPU runtime required)
import subprocess, sys
gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu.returncode != 0:
    print("No GPU detected. Switch to an L4 runtime first, or skip this section.")
else:
    print(gpu.stdout.split("\n")[8] if len(gpu.stdout.split("\n")) > 8 else gpu.stdout[:400])
    # !pip install -q --extra-index-url=https://pypi.nvidia.com cuml-cu12
    print("\nUncomment the pip line above to install cuML.")

In [ ]:
# Cell 2 of 2 - CPU vs GPU Random Forest training time
import time
from sklearn.ensemble import RandomForestRegressor

X = preprocessing.select_features(labelled)
y = labelled["calorie_target"].astype(float)
preprocessor = preprocessing.build_preprocessor().fit(X)
X_dense = preprocessor.transform(X).astype("float32")

started = time.perf_counter()
RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1).fit(X_dense, y)
cpu_seconds = time.perf_counter() - started
print(f"scikit-learn (CPU): {cpu_seconds:.2f} s")

try:
    from cuml.ensemble import RandomForestRegressor as cuRF
    started = time.perf_counter()
    cuRF(n_estimators=500, random_state=42).fit(X_dense, y.to_numpy("float32"))
    gpu_seconds = time.perf_counter() - started
    print(f"cuML (L4 GPU)    : {gpu_seconds:.2f} s")
    print(f"Speed-up         : {cpu_seconds / gpu_seconds:.2f}x")
    print("\nNote for the report: at this dataset size GPU kernel-launch overhead")
    print("can outweigh the parallelism, so a speed-up below 1.0x is a legitimate")
    print("finding worth discussing - not a failure.")
except ImportError:
    print("cuML not installed - skipping GPU benchmark.")

## 7. Figures produced

All figures are in `ml/reports/figures/` at 150 dpi, ready to drop into the
dissertation:

| File | Use in report |
|---|---|
| `eval_predicted_vs_actual.png` | Model accuracy |
| `eval_residual_distributions.png` | Error behaviour |
| `eval_learning_curves.png` | **Key evidence** for the LR-vs-RF discussion |
| `eval_feature_importance.png` | Which features drive predictions |
| `eval_meal_plan_accuracy.png` | Planner meets the ±5 % tolerance |
| `labels_*.png` | Methodology / label construction |
| `eda_*.png` | Dataset Review chapter |

**Next:** `05_export.ipynb`